In [14]:
import random
def generate_arithmetic_sequence(max_terms=3, max_digits=2, min_val=0, max_val=100):
    """
    生成一个算术表达式和对应的结果，用于 seq2seq 训练。
    返回: (input_str, target_str)
    """
    num_terms = random.randint(2, max_terms)
    operators = ['+', '-']
    tokens = []
    total = 0

    for i in range(num_terms):
        # 生成一个数字，位数不超过 max_digits
        num = random.randint(min_val, 10 ** max_digits - 1)
        if i == 0:
           
            total = num
        else:
            op = random.choice(operators)
            if op == '+':
                total += num
            else:
                total -= num
        if i>0:
            tokens.append(op)
        tokens.append(str(num))
       

    # 确保结果在合理范围（如果需要非负，可以加一个判断）
    # 这里允许负数，因为 seq2seq 应该能处理
    expr = ' '.join(tokens)
    result = str(total)
    return expr + ' = ', result

generate_arithmetic_sequence()

('79 + 17 + 34 = ', '130')

In [ ]:
class SymbolicArithmeticDataset(Dataset):
    def __init__(self, num_samples, max_terms=3, max_digits=2, min_val=0, max_val=100):
        self.samples = self._generate(num_samples, max_terms, max_digits, min_val, max_val)
        self.vocab = self._build_vocab()
        self.char2idx = {c: i for i, c in enumerate(self.vocab)}
        self.idx2char = {i: c for i, c in enumerate(self.vocab)}
        self.vocab_size = len(self.vocab)
        self.max_len = max( len(s['input']) for s in self.samples ) + 10

    def _build_vocab(self):
        # 字符表：数字、运算符、等号、空格、特殊符号
        return ['0','1','2','3','4','5','6','7','8','9','+','-','=',' ', '<SOS>', '<EOS>', '<PAD>']

    def _generate(self, num_samples, max_terms, max_digits, min_val, max_val):
        samples = []
        for _ in range(num_samples):
            expr, result = self._generate_expression(max_terms, max_digits, min_val, max_val)
            samples.append({'input': expr + ' = ', 'output': result})
        return samples

    def __getitem__(self, idx):
        sample = self.samples[idx]
        in_seq = sample['input'] + '<EOS>'
        out_seq = '<SOS>' + sample['output'] + '<EOS>'
        in_ids = [self.char2idx[c] for c in in_seq]
        out_ids = [self.char2idx[c] for c in out_seq]
        # 填充到 max_len
        in_ids = in_ids + [self.char2idx['<PAD>']] * (self.max_len - len(in_ids))
        out_ids = out_ids + [self.char2idx['<PAD>']] * (self.max_len - len(out_ids))
        return {
            'input_ids': torch.tensor(in_ids, dtype=torch.long),
            'output_ids': torch.tensor(out_ids, dtype=torch.long),
            'in_len': len(in_seq),
            'out_len': len(out_seq)
        }